In [1]:
!ls



DNGO_bo.ipynb DNN_bo.ipynb  GPR_bo.ipynb


In [2]:
cd ~/MultiFidelity-ProcessOpt/Process/1. Code


/Users/k23070952/MultiFidelity-ProcessOpt/Process/1. Code


In [3]:
from ShortCutDesign import ShortCutDesign 
from RigorousDesign import RigorousDesign 
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np
import os

In [4]:

# ✅ 파일 경로 설정
results_file = '../3. Data/6000_gp_bo_data_251024.csv'
error_file = '../3. Data/6000_gp_bo_errors_251024.csv'

if not os.path.exists('../3. Data/'):
    os.makedirs('../3. Data/')

# ✅ Shortcut 및 Rigorous 결과 계산 함수 
def calculate_shortcut(_init_params):
    shortcut_model = ShortCutDesign()
    shortcut_results = shortcut_model.shortcut_results(_init_params)
    return shortcut_results

def calculate_rigorous(_init_params):
    shortcut_results = calculate_shortcut(_init_params)
    if shortcut_results['CAPEX'] == 0:
        return (shortcut_results, [0, 0, 0])

    if shortcut_results['CAPEX'] is None or np.isnan(shortcut_results['CAPEX']):
        return (shortcut_results, [0, 0, 0])

    RigorousCal = RigorousDesign()
    Rigorous_results = RigorousCal.func(_init_params, shortcut_results)
    return (shortcut_results, Rigorous_results)



In [5]:
# ✅ 평가 함수 (병렬 실행용)

# ✅ Shortcut 및 Rigorous 결과 계산 함수 
def calculate_shortcut(_init_params):
    shortcut_model = ShortCutDesign()
    shortcut_results = shortcut_model.shortcut_results(_init_params)
    return shortcut_results

def calculate_rigorous(_init_params):
    shortcut_results = calculate_shortcut(_init_params)
    if shortcut_results['CAPEX'] == 0:
        return (shortcut_results, [0, 0, 0])

    if shortcut_results['CAPEX'] is None or np.isnan(shortcut_results['CAPEX']):
        return (shortcut_results, [0, 0, 0])

    RigorousCal = RigorousDesign()
    Rigorous_results = RigorousCal.check_results(_init_params, shortcut_results)
    return (shortcut_results, Rigorous_results)


bounds = [
    Integer(1, 50, name="n1"),  # Extractor stages
    Real(0, 0.9999, name="Lr1"),
    Real(0, 0.9999, name="Hr1"),
    Real(0, 0.9999, name="Lr2"),
    Real(0, 0.9999, name="Hr2"),
    Real(273, 350, name="T_hex"),
    Real(0, 0.9999, name="Lr3"),
    Real(0, 0.9999, name="Hr3")
]

opt_history = []

@use_named_args(bounds)
def objective(**params):
    X = [params[key] for key in ["n1", "Lr1", "Hr1", "Lr2", "Hr2", "T_hex", "Lr3", "Hr3"]]
    try:
        # 계산 수행
        shortcut_obj, rigorous_obj = calculate_rigorous(X)

        msp = rigorous_obj[-1]
        
        # 결과 데이터 정리
        record = list(X) + [
            shortcut_obj['CAPEX'], shortcut_obj['OPEX'], shortcut_obj['AceticAcidWt'], shortcut_obj['SplitRatio'], 
            shortcut_obj['boilup_1'], shortcut_obj['N_stages_1'], shortcut_obj['feed_stage_1'], 
            shortcut_obj['boilup_2'],shortcut_obj['N_stages_2'],shortcut_obj['feed_stage_2'],
            shortcut_obj['boilup_3'],shortcut_obj['N_stages_3'],shortcut_obj['feed_stage_3'],shortcut_obj['shortcut_time'],
            rigorous_obj[0], rigorous_obj[1], rigorous_obj[2], rigorous_obj[3], msp, None
        ]

        # rigorous_result = (CAPEX, OPEX, AceticAcid_wt, time)
          # 예: CAPEX + OPEX

        # 실시간 CSV 저장
        df = pd.DataFrame([record])
        df.to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)

        return msp

    except Exception as e:
        # 실패한 경우에는 파라미터와 에러 메시지만 저장
        record = list(X) + [None]*21 + [str(e)]  # 앞에 8개 + 21개 + error
        df = pd.DataFrame([record])
        df.to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)
        
        return 10000

In [7]:


# Run Bayesian Optimization with GPR
res = gp_minimize(
    func=objective,
    dimensions=bounds,
    n_calls=150,
    n_initial_points=100,
    random_state=42,
    verbose=True
)



Iteration No: 1 started. Evaluating function at random point.
 ##### An instance of the 'BlackBox' class  has been initialised!
[40, 0.18341644638717722, 0.7796130311727423, 0.5967904729306924, 0.4457881695783059, 280.6980685179862, 0.4592029670766707, 0.333675240277908]
setting base model
setting model:  {'SplitRatio': 0.287, 'boilup_1': 4.48, 'N_stages_1': 9, 'feed_stage_1': 7, 'boilup_2': 4.48, 'N_stages_2': 9, 'feed_stage_2': 7, 'boilup_3': 4.48, 'N_stages_3': 9, 'feed_stage_3': 7}
10000.0
 ##### An instance of the 'RigorousDesign' class  has been initialised!
[40, 0.18341644638717722, 0.7796130311727423, 0.5967904729306924, 0.4457881695783059, 280.6980685179862, 0.4592029670766707, 0.333675240277908]
setting model:  {'CAPEX': 1.6629, 'OPEX': 3.4506, 'AceticAcidWt': 0.04454200357104111, 'SplitRatio': 0.01380670611439842, 'boilup_1': 0.2437120001482147, 'N_stages_1': 4, 'feed_stage_1': 3, 'boilup_2': 1.5059112289316585, 'N_stages_2': 6, 'feed_stage_2': 6, 'boilup_3': 1.9836798236929

In [ ]:
opt_history

[{'params': {'n1': 42,
   'Lr1': 0.18341644638717722,
   'Hr1': 0.7796130311727423,
   'Lr2': 0.5967904729306924,
   'Hr2': 0.4457881695783059,
   'T_hex': 280.6980685179862,
   'Lr3': 0.4592029670766707,
   'Hr3': 0.333675240277908},
  'target': 100},
 {'params': {'n1': 16,
   'Lr1': 0.6508233841015582,
   'Hr1': 0.05640593786919756,
   'Lr2': 0.7219265723895982,
   'Hr2': 0.9384588537448488,
   'T_hex': 273.0599649697581,
   'Lr3': 0.9921123381352887,
   'Hr3': 0.6174197614767539},
  'target': 100},
 {'params': {'n1': 34,
   'Lr1': 0.007065598589195436,
   'Hr1': 0.02306011879891162,
   'Lr2': 0.5247221827923634,
   'Hr2': 0.39982098561808405,
   'T_hex': 276.5932560674484,
   'Lr3': 0.9736581432895752,
   'Hr3': 0.23274806329626127},
  'target': 100},
 {'params': {'n1': 14,
   'Lr1': 0.6183241707321541,
   'Hr1': 0.3824237450680361,
   'Lr2': 0.9831325627182077,
   'Hr2': 0.46671621695865523,
   'T_hex': 339.2154113186967,
   'Lr3': 0.6802395078339211,
   'Hr3': 0.45045420204434616}

In [14]:
# Load results
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df_results = pd.read_csv(results_file)

# 컬럼명 확인 (CSV 헤더가 없을 수 있음)
if df_results.columns[0] == '0':  # 헤더가 없는 경우
    column_names = ['n1', 'Lr1', 'Hr1', 'Lr2', 'Hr2', 'T_hex', 'Lr3', 'Hr3',
                    'sc_CAPEX', 'sc_OPEX', 'sc_AceticAcidWt', 'sc_SplitRatio',
                    'sc_boilup_1', 'sc_N_stages_1', 'sc_feed_stage_1',
                    'sc_boilup_2', 'sc_N_stages_2', 'sc_feed_stage_2',
                    'sc_boilup_3', 'sc_N_stages_3', 'sc_feed_stage_3', 'sc_time',
                    'rg_CAPEX', 'rg_OPEX', 'rg_AceticAcidWt', 'rg_time',
                    'MSP', 'error']
    df_results.columns = column_names

# MSP 값 추출
msp_values = df_results['MSP'].values

# 초기 샘플링과 BO 구간 분리
n_initial = 100
initial_msp = msp_values[:n_initial]
bo_msp = msp_values[n_initial:]

# Best-so-far 계산
best_so_far = np.minimum.accumulate(msp_values)
initial_best = best_so_far[:n_initial]
bo_best = best_so_far[n_initial:]

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Best-so-far curve
ax = axes[0, 0]
ax.plot(range(1, len(best_so_far) + 1), best_so_far, 'b-', linewidth=2, label='Best-so-far')
ax.axvline(x=n_initial, color='red', linestyle='--', linewidth=2, label='BO starts', alpha=0.7)
ax.fill_betweenx([ax.get_ylim()[0], ax.get_ylim()[1]], 0, n_initial, alpha=0.2, color='gray',
label='Initial sampling')
ax.set_xlabel('Evaluation Number', fontsize=12)
ax.set_ylabel('Best MSP Found ($/kg)', fontsize=12)
ax.set_title('GP-BO Optimization Progress', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# 2. All evaluations (Initial vs BO)
ax = axes[0, 1]
ax.scatter(range(1, n_initial + 1), initial_msp, c='gray', alpha=0.5, s=30, label=f'Initial   sampling (n={n_initial})')
ax.scatter(range(n_initial + 1, len(msp_values) + 1), bo_msp, c='blue', alpha=0.7, s=50,
marker='s', label=f'GP-BO (n={len(bo_msp)})')
ax.axvline(x=n_initial, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Evaluation Number', fontsize=12)
ax.set_ylabel('MSP ($/kg)', fontsize=12)
ax.set_title('All Evaluations: Initial vs BO', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. MSP Distribution comparison
ax = axes[1, 0]
ax.hist(initial_msp, bins=30, alpha=0.6, color='gray', label='Initial sampling', edgecolor='black')
ax.hist(bo_msp, bins=20, alpha=0.6, color='blue', label='GP-BO', edgecolor='black')
ax.axvline(x=np.min(msp_values), color='red', linestyle='--', linewidth=2, label=f'Best: {np.min(msp_values):.4f}')
ax.set_xlabel('MSP ($/kg)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('MSP Distribution: Initial vs BO', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 4. Improvement per iteration
ax = axes[1, 1]
improvements = -np.diff(best_so_far)  # Negative diff = improvement
improvements = np.concatenate([[0], improvements])  # First iteration has no improvement
colors = ['green' if imp > 0 else 'lightgray' for imp in improvements]
ax.bar(range(1, len(improvements) + 1), improvements, color=colors, alpha=0.7)
ax.axvline(x=n_initial, color='red', linestyle='--', linewidth=2, alpha=0.7, label='BO starts')
ax.set_xlabel('Evaluation Number', fontsize=12)
ax.set_ylabel('Improvement in MSP ($/kg)', fontsize=12)
ax.set_title('Per-Iteration Improvement', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# 타임스탬프 생성 (없으면)
from datetime import datetime
timestamp = datetime.now().strftime("%y%m%d_%H%M%S")
results_dir = '../3. Data'

plt.savefig(f'{results_dir}/gpr_bo_visualization_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
print("\n" + "=" * 80)
print("GP-BO OPTIMIZATION SUMMARY")
print("=" * 80)

print(f"\nInitial Sampling (n={n_initial}):")
print(f"  Best MSP: {np.min(initial_msp):.4f}")
print(f"  Mean MSP: {np.mean(initial_msp):.4f}")
print(f"  Std MSP: {np.std(initial_msp):.4f}")

print(f"\nGP-BO Phase (n={len(bo_msp)}):")
print(f"  Best MSP: {np.min(bo_msp):.4f}")
print(f"  Mean MSP: {np.mean(bo_msp):.4f}")
print(f"  Std MSP: {np.std(bo_msp):.4f}")

print(f"\nOverall (n={len(msp_values)}):")
print(f"  Best MSP: {np.min(msp_values):.4f}")
print(f"  Final MSP: {msp_values[-1]:.4f}")
print(f"  Improvement from initial best: {np.min(initial_msp) - np.min(msp_values):.4f}")
print(f"  Total evaluations: {len(msp_values)}")

# Best configuration
best_idx = np.argmin(msp_values)
print(f"\nBest Configuration (Evaluation #{best_idx + 1}):")
print(f"  n1: {df_results.iloc[best_idx]['n1']:.0f}")
print(f"  Lr1: {df_results.iloc[best_idx]['Lr1']:.4f}")
print(f"  Hr1: {df_results.iloc[best_idx]['Hr1']:.4f}")
print(f"  Lr2: {df_results.iloc[best_idx]['Lr2']:.4f}")
print(f"  Hr2: {df_results.iloc[best_idx]['Hr2']:.4f}")
print(f"  T_hex: {df_results.iloc[best_idx]['T_hex']:.2f} K")
print(f"  Lr3: {df_results.iloc[best_idx]['Lr3']:.4f}")
print(f"  Hr3: {df_results.iloc[best_idx]['Hr3']:.4f}")
print(f"  MSP: {df_results.iloc[best_idx]['MSP']:.4f} $/kg")


ValueError: Length mismatch: Expected axis has 30 elements, new values have 28 elements